In [0]:
import requests

url = "https://github.com/wmgeolab/geoBoundaries/raw/9469f09/releaseData/gbOpen/JOR/ADM1/geoBoundaries-JOR-ADM1_simplified.geojson"

response = requests.get(url)

print(response.status_code)
print(len(response.text))

In [0]:
import json
import pandas as pd

geojson = response.json()

features = geojson["features"]

print(len(features))
print(features[0].keys())
print(features[0]["properties"])

In [0]:
rows = []

for feature in features:
    props = feature["properties"]

    rows.append({
        "shape_name": props.get("shapeName"),
        "shape_id": props.get("shapeID"),
        "shape_type": props.get("shapeType"),
        "shape_group": props.get("shapeGroup"),
        "geometry_type": feature["geometry"]["type"],
        "geometry_json": json.dumps(feature["geometry"])
    })

geo_df = pd.DataFrame(rows)

display(geo_df)

In [0]:
spark_df = spark.createDataFrame(geo_df)

spark.sql("CREATE SCHEMA IF NOT EXISTS info_env_jordan.bronze")

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("info_env_jordan.bronze.geo_jordan_governorates")